# Text-to-SQL Avanzado: Corrección Automática y Esquemas Complejos

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexfazio/InteligenciaArtificial/blob/main/tutoriales/notebooks/text-to-sql/02-text-to-sql-avanzado.ipynb)

Implementa self-correction loops, schema-linking y manejo de fechas relativas para llevar la precisión del Text-to-SQL del 70% al 85%+.

In [ ]:
%pip install anthropic pandas --quiet

In [ ]:
import os
import re
import sqlite3
import pandas as pd
import anthropic
from datetime import date

client = anthropic.Anthropic()

# Base de datos con relaciones más complejas
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON')
conn.executescript("""
    CREATE TABLE regiones (id INTEGER PRIMARY KEY, nombre TEXT, pais TEXT);
    CREATE TABLE clientes (
        id INTEGER PRIMARY KEY, nombre TEXT, email TEXT,
        region_id INTEGER, fecha_registro TEXT,
        FOREIGN KEY (region_id) REFERENCES regiones(id)
    );
    CREATE TABLE categorias (id INTEGER PRIMARY KEY, nombre TEXT, margen REAL);
    CREATE TABLE productos (
        id INTEGER PRIMARY KEY, nombre TEXT, precio REAL, stock INTEGER,
        categoria_id INTEGER,
        FOREIGN KEY (categoria_id) REFERENCES categorias(id)
    );
    CREATE TABLE pedidos (
        id INTEGER PRIMARY KEY, cliente_id INTEGER,
        fecha TEXT, total REAL, estado TEXT,
        FOREIGN KEY (cliente_id) REFERENCES clientes(id)
    );
    INSERT INTO regiones VALUES (1,'Norte','España'),(2,'Sur','España'),(3,'Centro','España');
    INSERT INTO categorias VALUES (1,'Electrónica',0.30),(2,'Periféricos',0.45),(3,'Audio',0.40);
    INSERT INTO clientes VALUES
        (1,'Ana García','ana@mail.com',1,'2024-01-15'),
        (2,'Luis Martínez','luis@mail.com',2,'2024-02-20'),
        (3,'María López','maria@mail.com',3,'2024-03-10');
    INSERT INTO productos VALUES
        (1,'Laptop Pro',1299.99,15,1),(2,'Monitor 4K',449.99,30,1),
        (3,'Teclado',89.99,50,2),(4,'Auriculares',129.99,20,3);
    INSERT INTO pedidos VALUES
        (1,1,'2024-05-01',1299.99,'completado'),
        (2,2,'2024-05-03',539.98,'completado'),
        (3,3,'2024-05-15',169.98,'pendiente'),
        (4,1,'2024-06-01',449.99,'enviado'),
        (5,2,'2024-06-10',89.99,'completado');
""")
conn.commit()
print('BD con 5 tablas y relaciones FK creada.')

In [ ]:
def extraer_esquema_con_fk(conn):
    """Esquema compacto con anotaciones de foreign keys."""
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tablas = [row[0] for row in cursor.fetchall()]
    
    partes = []
    for tabla in tablas:
        cursor.execute(f'PRAGMA table_info({tabla})')
        columnas = {c[1]: c[2] for c in cursor.fetchall()}
        cursor.execute(f'PRAGMA foreign_key_list({tabla})')
        fks = {fk[3]: (fk[2], fk[4]) for fk in cursor.fetchall()}
        
        cols_desc = []
        for nombre, tipo in columnas.items():
            if nombre in fks:
                tabla_ref, col_ref = fks[nombre]
                cols_desc.append(f'{nombre} {tipo} -> {tabla_ref}.{col_ref}')
            else:
                cols_desc.append(f'{nombre} {tipo}')
        partes.append(f"{tabla}({', '.join(cols_desc)})")
    return '\n'.join(partes)

def agregar_contexto_temporal(esquema):
    """Añade la fecha actual y patrones de conversión al esquema."""
    hoy = date.today().isoformat()
    return esquema + f"""

Fecha actual: {hoy}
- 'hoy' -> DATE('{hoy}')
- 'este mes' -> strftime('%Y-%m', fecha) = strftime('%Y-%m', '{hoy}')
- 'el mes pasado' -> strftime('%Y-%m', fecha) = strftime('%Y-%m', DATE('{hoy}', '-1 month'))
- 'últimos 30 días' -> fecha >= DATE('{hoy}', '-30 days')
"""

esquema_base = extraer_esquema_con_fk(conn)
print('Esquema con FK:')
print(esquema_base)

In [ ]:
def schema_linking(pregunta, esquema_dict):
    """Identifica las tablas relevantes para la pregunta."""
    esquema_texto = '\n'.join(
        f"{tabla}: {', '.join(cols)}" for tabla, cols in esquema_dict.items()
    )
    respuesta = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=128,
        system='Dado un esquema y una pregunta, responde SOLO con los nombres de las tablas necesarias, separados por coma.',
        messages=[{'role': 'user', 'content': f'Esquema:\n{esquema_texto}\n\nPregunta: {pregunta}\n\n¿Qué tablas son necesarias?'}]
    )
    tablas = [t.strip() for t in respuesta.content[0].text.split(',')]
    return tablas

# Probar schema-linking
esquema_dict = {
    'regiones': ['id', 'nombre', 'pais'],
    'clientes': ['id', 'nombre', 'email', 'region_id', 'fecha_registro'],
    'categorias': ['id', 'nombre', 'margen'],
    'productos': ['id', 'nombre', 'precio', 'stock', 'categoria_id'],
    'pedidos': ['id', 'cliente_id', 'fecha', 'total', 'estado'],
}

preguntas_test = [
    '¿Cuántos clientes hay en cada región?',
    '¿Qué productos de electrónica tienen stock bajo?',
    '¿Cuántos pedidos completados hay?',
]

for p in preguntas_test:
    tablas = schema_linking(p, esquema_dict)
    print(f'Pregunta: {p}')
    print(f'Tablas relevantes: {tablas}\n')

In [ ]:
def generar_sql_con_correccion(pregunta, esquema, conn, max_intentos=3):
    """Self-correction loop: hasta max_intentos de corregir errores SQL."""
    historial_mensajes = []
    ultimo_error = None
    
    system_prompt = f"""Eres un experto en SQL para SQLite. Genera consultas SQL correctas.

Esquema:
{esquema}

Responde SOLO con la consulta SQL, sin markdown."""
    
    for intento in range(1, max_intentos + 1):
        if ultimo_error:
            contenido = f"""La consulta anterior falló:
Error: {ultimo_error}

Corrige el SQL para: {pregunta}"""
        else:
            contenido = pregunta
        
        historial_mensajes.append({'role': 'user', 'content': contenido})
        
        respuesta = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=512,
            system=system_prompt,
            messages=historial_mensajes
        )
        
        sql = respuesta.content[0].text.strip()
        if sql.startswith('```'):
            sql = '\n'.join(sql.split('\n')[1:-1]).strip()
        
        historial_mensajes.append({'role': 'assistant', 'content': sql})
        
        try:
            df = pd.read_sql_query(sql, conn)
            print(f'  [Intento {intento}] Éxito')
            return {'sql': sql, 'dataframe': df, 'intentos': intento, 'error': None}
        except Exception as e:
            ultimo_error = str(e)
            print(f'  [Intento {intento}] Error: {ultimo_error[:80]}')
    
    return {'sql': sql, 'dataframe': None, 'intentos': max_intentos, 'error': ultimo_error}

print('Función de self-correction definida.')

In [ ]:
# Clase AdvancedTextToSQL integrada
class AdvancedTextToSQL:
    def __init__(self, conn, max_intentos=3):
        self.conn = conn
        self.max_intentos = max_intentos
        self.esquema_dict = self._cargar_esquema_dict()
    
    def _cargar_esquema_dict(self):
        cursor = self.conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tablas = [row[0] for row in cursor.fetchall()]
        resultado = {}
        for tabla in tablas:
            cursor.execute(f'PRAGMA table_info({tabla})')
            resultado[tabla] = [col[1] for col in cursor.fetchall()]
        return resultado
    
    def query(self, pregunta):
        # Schema-linking para BD grandes, esquema completo para pequeñas
        if len(self.esquema_dict) > 8:
            tablas_relevantes = schema_linking(pregunta, self.esquema_dict)
            esquema_filtrado = {t: c for t, c in self.esquema_dict.items() if t in tablas_relevantes}
            esquema_str = '\n'.join(f"{t}({', '.join(c)})" for t, c in esquema_filtrado.items())
        else:
            esquema_str = extraer_esquema_con_fk(self.conn)
        
        # Agregar contexto temporal
        esquema_str = agregar_contexto_temporal(esquema_str)
        
        resultado = generar_sql_con_correccion(
            pregunta, esquema_str, self.conn, self.max_intentos
        )
        
        if resultado['dataframe'] is not None:
            df = resultado['dataframe']
            respuesta = client.messages.create(
                model='claude-haiku-4-5-20251001',
                max_tokens=256,
                messages=[{'role': 'user', 'content': f'Pregunta: {pregunta}\nResultado:\n{df.to_string(index=False)}\n\nResponde en una oración clara en español.'}]
            )
            resultado['respuesta'] = respuesta.content[0].text.strip()
        
        return resultado

sistema = AdvancedTextToSQL(conn)
print('Sistema avanzado inicializado.')

In [ ]:
# Demo: preguntas con JOINs, fechas relativas y agregaciones
preguntas_avanzadas = [
    '¿Cuántos pedidos completados hay por región del cliente?',
    '¿Qué categorías de productos tienen el mayor promedio de precio?',
    '¿Cuántos pedidos se hicieron en los últimos 60 días?',
]

for pregunta in preguntas_avanzadas:
    print(f'\nPregunta: {pregunta}')
    res = sistema.query(pregunta)
    if res['error']:
        print(f'Error final: {res["error"]}')
    else:
        print(f'SQL ({res["intentos"]} intento/s): {res["sql"]}')
        if res.get('respuesta'):
            print(f'Respuesta: {res["respuesta"]}')

In [ ]:
# Evaluación: Execution Accuracy vs Exact Match
GOLDEN_SET = [
    {'pregunta': '¿Cuántos clientes hay?', 'sql': 'SELECT COUNT(*) AS total FROM clientes'},
    {'pregunta': '¿Cuál es el producto más caro?', 'sql': 'SELECT nombre, precio FROM productos ORDER BY precio DESC LIMIT 1'},
    {'pregunta': '¿Cuántos pedidos por estado?', 'sql': 'SELECT estado, COUNT(*) AS total FROM pedidos GROUP BY estado'},
]

def exact_match(sql_gen, sql_esp):
    normalizar = lambda s: ' '.join(s.lower().split())
    return normalizar(sql_gen) == normalizar(sql_esp)

correctos_em = 0
for caso in GOLDEN_SET:
    sql_generado = ''
    try:
        res = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=256,
            system=f'Esquema:\n{extraer_esquema_con_fk(conn)}\nResponde solo SQL.',
            messages=[{'role': 'user', 'content': caso['pregunta']}]
        )
        sql_generado = res.content[0].text.strip()
        if sql_generado.startswith('```'):
            sql_generado = '\n'.join(sql_generado.split('\n')[1:-1]).strip()
    except Exception as e:
        print(f'Error: {e}')
        continue
    
    em = exact_match(sql_generado, caso['sql'])
    if em:
        correctos_em += 1
    print(f'{"✓" if em else "~"} Exact Match: {caso["pregunta"][:50]}')

print(f'\nExact Match Score: {correctos_em}/{len(GOLDEN_SET)} = {correctos_em/len(GOLDEN_SET)*100:.0f}%')
print('(Nota: EM es muy estricta. Execution Accuracy es la métrica recomendada.)')

## Resumen

Implementaste tres técnicas avanzadas:

1. **Self-correction**: hasta 3 intentos de corregir SQL con el mensaje de error real
2. **Schema-linking**: filtrar tablas relevantes para ahorrar tokens con BD grandes
3. **Fechas relativas**: inyectar `CURRENT_DATE` y patrones de conversión

**Siguiente**: `03-agente-analista-sql.ipynb` — Agente con SQL + gráficas + insights